<a href="https://colab.research.google.com/github/owennoimor/bco7006_practice/blob/main/epss_v2_with_chatgpt_submit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **VERSION 2 AFTER CHATGPT**

#**Assessment 3: Programming Solutions**

## Problem Statement:

The current enrolment process at the university is time-consuming, inefficient, and does not effectively match students with appropriate courses. This results in lower satisfaction and retention rates among students, which can harm the university's reputation and financial stability.

## Solution:
To address this problem, the university should implement an enrollment process Screening System (EPSS) that checks the course structure and students' enrolment background before enrolling them. This system can be achieved by leveraging programming algorithms that analyse student data to recommend courses that align with their interests and academic abilities.

#Condition 1: Course Structure

 The course structure is submitted below. The unit’s prerequisite for the enrolment should be considered.

- unit_name

- unit_code

- pre-req/ condition

- semest

**Step 1 - Catalogue + one student**

**Prompt**

How would you refactor this to use @dataclass for the student and enrolment records?


Main improvements:

- This makes the code more readable, organised, and easier by using @dataclass

In [1]:
from dataclasses import dataclass

# Course structure sample data
catalogue = {
    "BCO7006": {"name": "Coding for BA", "prereqs": [], "semester": 1, "female_only": 0},
    "BCO7000": {"name": "Biz analytics", "prereqs": [], "semester": 1, "female_only": 0},
    "BCO6008": {"name": "Predictive Analytics", "prereqs": ["BCO7000"], "semester": 2, "female_only": 0},
    "BCO7007": {"name": "Machine Learning", "prereqs": ["BCO7006", "BCO7000"], "semester": 2, "female_only": 0},
    "WOM1000": {"name": "Women in STEM", "prereqs": [], "semester": 0, "female_only": 1}
}


@dataclass
class Student:
    st_name: str
    st_id: str
    gender: str
    unit_passed: list


@dataclass
class EnrolmentRecord:
    st_id: str
    unit_code: str
    status: str
    reason: str


# Student sample data
alice = Student(
    st_name="Alice",
    st_id="S001",
    gender="Female",
    unit_passed=["BCO7000"]
)

# Example enrolment record
record1 = EnrolmentRecord(
    st_id="S001",
    unit_code="BCO6008",
    status="Enrolled",
    reason="Prerequisite met"
)

print(alice)
print(record1)

Student(st_name='Alice', st_id='S001', gender='Female', unit_passed=['BCO7000'])
EnrolmentRecord(st_id='S001', unit_code='BCO6008', status='Enrolled', reason='Prerequisite met')


In [2]:
alice.st_name

'Alice'

#Condition 2: Students Enrolment background

The students are allowed to enrol in two units however the program should check the students’ enrolment background.
Here is a table as a sample of an input.

- st_name

- st_id

- gender

- unit_passed

- semester

**Step 2 — One screening function**

**Prompt**

How can I improve this enrolment checking function while keeping logic?

Main improvements:

- cleaner spacing: unit_code, catalogue
-  clearer variable name: prereq instead of p
- better message formatting: BCO6008: Predictive Analytics
- safer/clearer condition: unit["female_only"] == 1 instead of just unit["female_only"]

In [3]:
def can_enrol(student, unit_code, catalogue):
    """Return (ok, message) for a single unit."""

    if unit_code not in catalogue:
        return False, f"Unit {unit_code} not found"

    unit = catalogue[unit_code]

    # Check prerequisites
    missing = [
        prereq for prereq in unit["prereqs"]
        if prereq not in student["unit_passed"]
    ]

    if missing:
        return False, f"Cannot enrol in {unit_code}. Missing prereqs: {', '.join(missing)}"

    # Check female-only condition
    if unit["female_only"] == 1 and student["gender"].lower() != "female":
        return False, f"Cannot enrol in {unit_code}. This unit is only for female students"

    return True, f"Enrolment OK for {unit_code}: {unit['name']}"

#Condition 3: Enrolment process
Your program should check if the students are eligible for enrolment, otherwise to send an appropriate message and advice to the student
Enrolment as an output:

- enrol_code

- Unit_code

- St_id

- semester

some units have special consition(s)

**Step 3 — The loop over requested units**

**Prompt**

Can you fix and improve this process_request function so it checks the 2-unit limit, loops through each requested unit, calls can_enrol, creates enrolment records, and returns the updated next_code?

Main improvements:

- len(requested_units>2) fixed to len(requested_units) > 2
- moved the for loop outside the if
- changed if not ok to else
- removed early return after one failed unit
- returns next_code after all units are checked

In [4]:
def process_request(student, requested_units, semester, catalogue, enrolments, next_code):
    """Process an enrolment request, update enrolments, and return the next code."""

    print(f"Processing request for {student['st_name']}")

    # Cap checking = no more than 2 units
    if len(requested_units) > 2:
        print(f"Rejecting request for {student['st_name']}: maximum 2 units allowed")
        return next_code

    for unit_code in requested_units:
        ok, message = can_enrol(student, unit_code, catalogue)

        if ok:
            record = {
                "enrol_code": next_code,
                "unit_code": unit_code,
                "st_id": student["st_id"],
                "semester": semester
            }

            enrolments.append(record)

            print(f"OK: {message} -> enrol_code: {next_code}")

            next_code += 1

        else:
            print(f"Fail: {message}")

    return next_code

**Step 4 — Wrap in a class**

**Prompt**

Review this code for readability and suggest improvements without changing the logic.

Main improvements:

- Remove unnecessary comments
- Use clear variable names
- Make the rejection rule easier to read
- Fix the class docstring


In [5]:
class EnrolmentSystemMin:
    """Minimum version of the enrolment screening system."""

    MAX_UNITS = 2

    def __init__(self, catalogue, enrolments=None, next_code=1001):
        self.catalogue = catalogue
        self.enrolments = enrolments if enrolments is not None else []
        self.next_code = next_code

    def check_student_eligibility(self, student, requested_units):
        """
        Check whether a student can enrol in each requested unit.

        Returns a list of (ok, message) tuples.
        """

        results = []

        if len(requested_units) > self.MAX_UNITS:
            message = (
                f"Cannot enrol in more than {self.MAX_UNITS} units. "
                f"Requested: {len(requested_units)}"
            )
            return [(False, message)]

        for unit_code in requested_units:
            ok, message = can_enrol(student, unit_code, self.catalogue)
            results.append((ok, message))

        return results

    def enrol_student(self, student, requested_units, semester):
        """
        Attempt to enrol a student in requested units.

        Returns True if all units are enrolled successfully,
        otherwise returns False.
        """

        print(f"Processing enrolment request for {student['st_name']}...")

        eligibility_results = self.check_student_eligibility(
            student,
            requested_units
        )

        request_too_large = (
            len(eligibility_results) == 1
            and not eligibility_results[0][0]
            and len(requested_units) > self.MAX_UNITS
        )

        if request_too_large:
            print(f"Rejected: {eligibility_results[0][1]}")
            return False

        all_ok = True

        for unit_code, (ok, message) in zip(requested_units, eligibility_results):
            if ok:
                record = {
                    "enrol_code": self.next_code,
                    "unit_code": unit_code,
                    "st_id": student["st_id"],
                    "semester": semester
                }

                self.enrolments.append(record)

                print(
                    f"Enrolment OK for {unit_code}: {message} "
                    f"(Enrol Code: {self.next_code})"
                )

                self.next_code += 1

            else:
                print(f"Enrolment FAILED for {unit_code}: {message}")
                all_ok = False

        return all_ok

**Step 5 — Interactive input**

**Prompt**

how can i improve enrolment system to be advanced by using input() to collect a student id and requested subject then all logics will be checked automatically?

Main Improvement:

-	Create input() function
- Program becomes interactive
-	User can enter their own student ID and subjects
-	Logic checks rules automatically


In [6]:
def find_student(student_id, students):
    for student in students:
        if student["st_id"] == student_id:
            return student
    return None


def run_enrolment_system(system, students):
    print("Welcome to the Enrolment System")

    student_id = input("Enter student ID: ").strip().upper()

    student = find_student(student_id, students)

    if student is None:
        print("Student not found")
        return

    print(f"Student found: {student['st_name']}")

    units_text = input("Enter requested unit codes separated by commas: ")
    requested_units = [
        unit.strip().upper()
        for unit in units_text.split(",")
    ]

    semester = int(input("Enter semester number: "))

    success = system.enrol_student(
        student,
        requested_units,
        semester
    )

    if success:
        print("All requested units enrolled successfully")
    else:
        print("Some units were rejected")

    print("Current enrolment records:")
    for record in system.enrolments:
        print(record)

#Test scenarios

In [7]:
#Student Sample
students = [
    {"st_name": "Alice", "st_id": "S001", "gender": "Female", "unit_passed": ["BCO7000"]},
    {"st_name": "Bob", "st_id": "S002", "gender": "Male", "unit_passed": []},
    {"st_name": "Owen","st_id":"S003","gender":"Male","unit_passed":["BCO7000", "BCO7006"]}
]

Scenario 1: Student has all prereqs, requests 2 valid units



Owen (S003) enrols BCO6008 and BCO7007 in  semester 2

In [ ]:
print("- Scenario 1 -")
# Expected: both enrol

system = EnrolmentSystemMin(catalogue)
run_enrolment_system(system, students)

- Scenario 1 -
Welcome to the Enrolment System
Enter student ID: s003
Student found: Owen
Enter requested unit codes separated by commas: bco6008, bco7007
Enter semester number: 2
Processing enrolment request for Owen...
Enrolment OK for BCO6008: Enrolment OK for BCO6008: Predictive Analytics (Enrol Code: 1001)
Enrolment OK for BCO7007: Enrolment OK for BCO7007: Machine Learning (Enrol Code: 1002)
All requested units enrolled successfully
Current enrolment records:
{'enrol_code': 1001, 'unit_code': 'BCO6008', 'st_id': 'S003', 'semester': 2}
{'enrol_code': 1002, 'unit_code': 'BCO7007', 'st_id': 'S003', 'semester': 2}


Scenario 2: Student missing one prereq

Alice (S001) enrols BCO6008 and BCO7007 in semester 2

In [ ]:
print("- Scenario 2 -")
# Expected: one enrols, one rejected

system = EnrolmentSystemMin(catalogue)
run_enrolment_system(system, students)

- Scenario 2 -
Welcome to the Enrolment System
Enter student ID: s001
Student found: Alice
Enter requested unit codes separated by commas: bco6008, bco7007
Enter semester number: 2
Processing enrolment request for Alice...
Enrolment OK for BCO6008: Enrolment OK for BCO6008: Predictive Analytics (Enrol Code: 1001)
Enrolment FAILED for BCO7007: Cannot enrol in BCO7007. Missing prereqs: BCO7006
Some units were rejected
Current enrolment records:
{'enrol_code': 1001, 'unit_code': 'BCO6008', 'st_id': 'S001', 'semester': 2}


Scenario 3: Student has no prereqs, requests two advanced units

Bob (S002) enrols BCO6008 and BCO7007 in semester 2

In [ ]:
print("- Scenario 3 -")
# Expected: both rejected

system = EnrolmentSystemMin(catalogue)
run_enrolment_system(system, students)

- Scenario 3 -
Welcome to the Enrolment System
Enter student ID: s002
Student found: Bob
Enter requested unit codes separated by commas: bco6008, bco7007
Enter semester number: 2
Processing enrolment request for Bob...
Enrolment FAILED for BCO6008: Cannot enrol in BCO6008. Missing prereqs: BCO7000
Enrolment FAILED for BCO7007: Cannot enrol in BCO7007. Missing prereqs: BCO7006, BCO7000
Some units were rejected
Current enrolment records:


Scenario 4: Female student requests WOM1000

Alice (S001) enrols WOM1000 in semester 3

In [ ]:
print("- Scenario 4 -")
# Expected: enrol

system = EnrolmentSystemMin(catalogue)
run_enrolment_system(system, students)

- Scenario 4 -
Welcome to the Enrolment System
Enter student ID: s001
Student found: Alice
Enter requested unit codes separated by commas: wom1000
Enter semester number: 3
Processing enrolment request for Alice...
Enrolment OK for WOM1000: Enrolment OK for WOM1000: Women in STEM (Enrol Code: 1001)
All requested units enrolled successfully
Current enrolment records:
{'enrol_code': 1001, 'unit_code': 'WOM1000', 'st_id': 'S001', 'semester': 3}


Scenario 5: 	Male student requests WOM1000

Owen (S003) enrols WOM1000 in semester 3

In [ ]:
print("- Scenario 5 -")
# Expected: rejected

system = EnrolmentSystemMin(catalogue)
run_enrolment_system(system, students)

- Scenario 5 -
Welcome to the Enrolment System
Enter student ID: s003
Student found: Owen
Enter requested unit codes separated by commas: wom1000
Enter semester number: 3
Processing enrolment request for Owen...
Enrolment FAILED for WOM1000: Cannot enrol in WOM1000. This unit is only for female students
Some units were rejected
Current enrolment records:


Scenario 6: Any student requests 3 units


Bob (S002) enrols BCO6008, BCO7007 and WOM1000 in semester 2

In [ ]:
print("- Scenario 6 -")
# Expected: rejected

system = EnrolmentSystemMin(catalogue)
run_enrolment_system(system, students)

- Scenario 6 -
Welcome to the Enrolment System
Enter student ID: s002
Student found: Bob
Enter requested unit codes separated by commas: bco6008, bco7007, wom1000
Enter semester number: 2
Processing enrolment request for Bob...
Rejected: Cannot enrol in more than 2 units. Requested: 3
Some units were rejected
Current enrolment records:
